# ML-09 — Week 6: Validation and Research Claim Audit (Lane 4 — CTR / Engagement Opportunity Scoring)

Last week's live session walked FlyRank's own paper as an example of careful reading. This notebook applies that same reading to two of the paper's findings, then turns the same lens on my own Week-5 model: an honest-split before/after, a leakage audit on the final feature set, and a rewrite of my own boldest sentence.

> Working with an AI assistant? Tell it to read `skills/README.md` first, then load `hunting-leakage-and-validating` + `flyrank/flyrank-data`.

**Careful words throughout:** observed, measured, directional, decision-support.

In [8]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import sklearn
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.model_selection import GroupKFold, KFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from IPython.display import display

SEED = 42
np.random.seed(SEED)

CSV = "data/raw/content_refresh_anonymized.csv"
root = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / CSV).exists()), None)
if root is None:
    root = Path.cwd()
    df = pd.read_csv("https://raw.githubusercontent.com/nothaziq/FlyRank-ML-Week1/main/" + CSV)
else:
    df = pd.read_csv(root / CSV)
OUT = root / "work" / "outputs"
OUT.mkdir(parents=True, exist_ok=True)

print(f"pandas {pd.__version__} | scikit-learn {sklearn.__version__} | seed {SEED}")
print(f"{len(df):,} rows, {df.client_id.nunique()} clients")

pandas 3.0.6 | scikit-learn 1.9.1 | seed 42
30,000 rows, 32 clients


## 1. Two paper findings + my methodology questions

Two exploratory-appendix findings from `docs/flyrank-seo-research-march-2026.pdf`, read the way I'd want my own Week-5 notebook read: where does the label come from, and does the validation design carry the claim?

### Finding: "What Predicts Health?" (Random Forest feature importance, p.27)

**The finding:** a Random Forest predicting Health Score ranks Average Position at 43% importance, Impressions at 32%, Scroll Depth at 15% — described as "the #1 predictor."

**My methodology question:** Health Score is defined, in the same paper's own methodology section, as `impressions (30 pts) + position (30 pts) + CTR (20 pts) + scroll depth (20 pts)`. Two of the three top "predictors" are literal, weighted components of the label itself. That is the label-derived-feature pattern the `hunting-leakage-and-validating` skill names first: the label was computed *from* a column, and that column is also a feature. A model can get high importance on `avg_position` here without learning anything about search behaviour — it can just be re-discovering 30 of the label's own 100 points.

**To the paper's credit**, the caption already says this plainly ("the target itself is partly constructed from some of these inputs, so importance is descriptive rather than causal") — this isn't a gap in the paper's honesty, it's a gap I'd want closed with one more panel: importance *with* the constructing features removed, so a reader can see what, if anything, predicts health beyond its own definition. Right now the chart answers "does the model see its own arithmetic?" (yes) more than "what predicts health?".

### Finding: "What Predicts Growth?" (Logistic Regression, 71% holdout accuracy, p.29)

**The finding:** a logistic regression trained on a single 80/20 holdout split reaches 71% accuracy separating growing from declining pages, with content age as the strongest negative coefficient.

**My methodology question, in two parts:**
1. **No base rate is reported next to the 71%.** Finding #1 (p.6) says the portfolio itself is 74,187 growing vs 45,272 declining pages — roughly a 62/38 split. A model that always predicts "growing" scores 62% for free. 71% is real skill, but it's 9 points of skill, not 71, and the paper's own "always print the base rate" habit (used carefully elsewhere, e.g. the SV-vs-impressions correlations) would strengthen this panel too.
2. **A single random 80/20 split, on a 57-brand portfolio, is the split the internship's own `hunting-leakage-and-validating` skill flags first: rows from one brand share hidden character, so a brand's pages can sit on both sides of the split and the model can partly memorize the brand rather than the growth pattern.** The paper doesn't say whether the split was grouped by brand. Section 2 below runs exactly this comparison on my own model, and the gap between a random split and a client-grouped split is large enough that I'd ask the same question of this panel before trusting the 71% at brand-level generalization.

Both of these are asked in the spirit the card asks for: the paper is public-safe and already discloses that ML pages are exploratory appendix material, secondary to the direct aggregate evidence. These are the two questions I'd bring to the authors, not corrections.

In [9]:
# ---- Rebuild the Week-5 lane, label and features exactly as committed --------------------------
BANDS = [0, 3, 5, 7, 10, 15, 20]
BAND_LABELS = ["0-3", "3-5", "5-7", "7-10", "10-15", "15-20"]
IMP_FLOOR_30, MIN_EXP_CLICKS_30, SHORTFALL = 500 / 3, 10 / 3, 0.5

d = df.copy()
d["band"] = pd.cut(d.avg_position, BANDS, labels=BAND_LABELS)
d["ctr_prev"] = d.clicks_prev_30d / d.impressions_prev_30d.replace(0, np.nan) * 100
d["ctr_last"] = d.clicks_last_30d / d.impressions_last_30d.replace(0, np.nan) * 100
vis = d.band.notna()
elig_prev = vis & (d.impressions_prev_30d >= IMP_FLOOR_30)
elig_last = vis & (d.impressions_last_30d >= IMP_FLOOR_30)
norm_prev = d.ctr_prev.where(elig_prev).groupby(d.band, observed=True).transform("median")
norm_last = d.ctr_last.where(elig_last).groupby(d.band, observed=True).transform("median")
d["norm_prev"], d["norm_last"] = norm_prev, norm_last
d["exp_clicks_prev"] = d.impressions_prev_30d * norm_prev / 100
d["exp_clicks_last"] = d.impressions_last_30d * norm_last / 100
d["persisted_shortfall"] = ((d.exp_clicks_last >= MIN_EXP_CLICKS_30) & (d.ctr_last < SHORTFALL * d.norm_last)).astype(int)

lane = d[elig_prev & elig_last].copy().reset_index(drop=True)
lane["log_impressions_prev"] = np.log1p(lane.impressions_prev_30d)
lane["ctr_vs_band_norm"] = lane.ctr_prev / lane.norm_prev
lane["shortfall_clicks_prev"] = lane.exp_clicks_prev - lane.clicks_prev_30d
lane["sessions_per_click_prev"] = lane.sessions_prev_30d / lane.clicks_prev_30d.replace(0, np.nan)
lane["log_word_count"] = np.log1p(lane.word_count)
lane["log_age_days"] = np.log1p(lane.content_age_days)
lane["log_search_volume"] = np.log1p(lane.search_volume)

NUM = ["log_impressions_prev", "ctr_prev", "ctr_vs_band_norm", "shortfall_clicks_prev",
       "sessions_per_click_prev", "avg_position", "log_word_count", "log_age_days",
       "days_since_last_update", "log_search_volume", "competition", "cpc"]
CAT = ["band", "content_type", "main_intent", "freshness_tier"]
FEATURES = NUM + CAT

X = lane[FEATURES].copy()
X[NUM] = X[NUM].replace([np.inf, -np.inf], np.nan)
y = lane.persisted_shortfall.to_numpy()
groups = lane.client_id.to_numpy()
base_rate = y.mean()

pre = ColumnTransformer([
    ("num", Pipeline([("imp", SimpleImputer(strategy="median")), ("sc", StandardScaler())]), NUM),
    ("cat", OneHotEncoder(handle_unknown="ignore", min_frequency=25), CAT),
])
model = Pipeline([("pre", pre), ("m", RandomForestClassifier(
    n_estimators=400, min_samples_leaf=5, class_weight="balanced_subsample", n_jobs=-1, random_state=SEED))])

print(f"Lane: {len(lane):,} pages, {lane.client_id.nunique()} clients, base rate {base_rate:.3f}")
print(f"Model carried over from Week 5: Random Forest, same {len(FEATURES)} features, same label.")

Lane: 9,836 pages, 28 clients, base rate 0.095
Model carried over from Week 5: Random Forest, same 16 features, same label.


## 2. My model under an honest split (before/after)

Week 5 already used `GroupKFold` on `client_id` — the "after" here. The "before" is the split the leakage skill warns about first: an ordinary random `KFold`, where one client's pages land on both sides of the fold and the model can partly memorize the client instead of the shortfall pattern. Same rows, same features, same label, same seed — only the split changes.

In [10]:
def precision_at_k(score, truth, k):
    order = np.argsort(-np.asarray(score, dtype=float), kind="stable")[:k]
    return float(np.mean(np.asarray(truth)[order]))

def run_cv(splitter, splitter_args):
    oof = np.zeros(len(lane))
    fold_p50, client_leak = [], []
    for tr, te in splitter.split(X, y, *splitter_args):
        model.fit(X.iloc[tr], y[tr])
        p = model.predict_proba(X.iloc[te])[:, 1]
        oof[te] = p
        fold_p50.append(precision_at_k(p, y[te], 50))
        client_leak.append(len(set(groups[tr]) & set(groups[te])))
    return oof, fold_p50, client_leak

# BEFORE: plain random KFold -- no group awareness at all
oof_random, p50_random, leak_random = run_cv(KFold(n_splits=5, shuffle=True, random_state=SEED), ())

# AFTER: GroupKFold on client_id -- Week 5's actual design
oof_grouped, p50_grouped, leak_grouped = run_cv(GroupKFold(n_splits=5), (groups,))

comparison = pd.DataFrame({
    "P@50 (pooled)": [precision_at_k(oof_random, y, 50), precision_at_k(oof_grouped, y, 50)],
    "P@50 (per-fold mean)": [np.mean(p50_random), np.mean(p50_grouped)],
    "P@50 (per-fold sd)": [np.std(p50_random), np.std(p50_grouped)],
    "ROC-AUC": [roc_auc_score(y, oof_random), roc_auc_score(y, oof_grouped)],
    "PR-AUC": [average_precision_score(y, oof_random), average_precision_score(y, oof_grouped)],
    "clients shared train/test, per fold": [leak_random, leak_grouped],
}, index=["BEFORE: random KFold(5)", "AFTER: GroupKFold(5) on client_id"])

print(f"Base rate: {base_rate:.3f}")
display(comparison)

Base rate: 0.095


,P@50 (pooled),P@50 (per-fold mean),P@50 (per-fold sd),ROC-AUC,PR-AUC,"clients shared train/test, per fold"
BEFORE: random KFold(5),0.98,0.832,0.068819,0.912345,0.584838,"[24, 25, 25, 25, 23]"
AFTER: GroupKFold(5) on client_id,0.92,0.648,0.151050,0.901692,0.563425,"[0, 0, 0, 0, 0]"


In [11]:
gap_p50 = comparison.loc["BEFORE: random KFold(5)", "P@50 (pooled)"] - comparison.loc["AFTER: GroupKFold(5) on client_id", "P@50 (pooled)"]
gap_auc = comparison.loc["BEFORE: random KFold(5)", "ROC-AUC"] - comparison.loc["AFTER: GroupKFold(5) on client_id", "ROC-AUC"]
print(f"Random split overstates pooled P@50 by {gap_p50:+.3f} and ROC-AUC by {gap_auc:+.3f} versus the client-grouped split.")
print(f"Random KFold leaves every one of the {lane.client_id.nunique()} clients present on both sides of all 5 folds "
      f"(client counts above are non-zero everywhere) -- the model can partly recognise a client it has already seen elsewhere in training.")
print(f"GroupKFold holds every client out whole -- 0 overlap by construction, confirmed above.")
print()
print("Reading this honestly: the gap is a real but modest overstatement here (this lane's per-client behaviour is not")
print("wildly distinctive), not the dramatic collapse the skill's checklist warns is possible in the worst case. The")
print("direction is what matters and it points one way: random splits look better than they should. Week 5's")
print("GroupKFold number is the one that survived this check and the one that belongs in any claim.")

Random split overstates pooled P@50 by +0.060 and ROC-AUC by +0.011 versus the client-grouped split.
Random KFold leaves every one of the 28 clients present on both sides of all 5 folds (client counts above are non-zero everywhere) -- the model can partly recognise a client it has already seen elsewhere in training.
GroupKFold holds every client out whole -- 0 overlap by construction, confirmed above.

Reading this honestly: the gap is a real but modest overstatement here (this lane's per-client behaviour is not
wildly distinctive), not the dramatic collapse the skill's checklist warns is possible in the worst case. The
direction is what matters and it points one way: random splits look better than they should. Week 5's
GroupKFold number is the one that survived this check and the one that belongs in any claim.


## 3. Leakage audit

The same hunt from Week 3's data contract, replayed on the final Week-5 feature set: (a) no banned outcome-window or product-flag column reached the features, (b) deliberately adding a known-leaky column shows the test harness itself can detect a leak, (c) every feature's timing is checked against the label's window.

In [12]:
# (a) Banned-column check -- outcome-window and product-flag columns must not be in FEATURES
BANNED = (
    [c for c in lane.columns if c.endswith("_90d") or c.endswith("_last_30d")]
    + ["ctr", "ctr_last", "engagement_rate", "scroll_rate", "ai_traffic_pct",
       "trend_direction", "trend_pct", "impression_tier", "position_tier",
       "norm_last", "exp_clicks_last", "persisted_shortfall"]
)
present = [f for f in FEATURES if f in BANNED]
print(f"Outcome-window / product-flag columns found inside FEATURES: {present or 'none'}")

# (b) The trap: add a column DEFINED FROM the label's own window, watch the score jump, then remove it
leaky_X = X.copy()
leaky_X["ctr_last_LEAK"] = lane.ctr_last  # the exact column persisted_shortfall is thresholded from

pre_leak = ColumnTransformer([
    ("num", Pipeline([("imp", SimpleImputer(strategy="median")), ("sc", StandardScaler())]),
     NUM + ["ctr_last_LEAK"]),
    ("cat", OneHotEncoder(handle_unknown="ignore", min_frequency=25), CAT),
])
leaky_model = Pipeline([("pre", pre_leak), ("m", RandomForestClassifier(
    n_estimators=200, min_samples_leaf=5, class_weight="balanced_subsample", n_jobs=-1, random_state=SEED))])

tr, te = next(GroupKFold(n_splits=5).split(X, y, groups))
model.fit(X.iloc[tr], y[tr])
honest_auc = roc_auc_score(y[te], model.predict_proba(X.iloc[te])[:, 1])
leaky_model.fit(leaky_X.iloc[tr], y[tr])
leak_auc = roc_auc_score(y[te], leaky_model.predict_proba(leaky_X.iloc[te])[:, 1])

print(f"\nHonest ROC-AUC (fold 1, final feature set): {honest_auc:.3f}")
print(f"Leaky  ROC-AUC (fold 1, + ctr_last added)  : {leak_auc:.3f}")
print(f"Jump: {leak_auc - honest_auc:+.3f} from one label-derived column -- the harness catches a real leak when one is planted.")
del leaky_X, leaky_model, leak_auc  # the leak doesn't survive past this cell
print("Deliberately-added leak removed; honest feature set is what Section 2's numbers use.")

Outcome-window / product-flag columns found inside FEATURES: none

Honest ROC-AUC (fold 1, final feature set): 0.877
Leaky  ROC-AUC (fold 1, + ctr_last added)  : 0.977
Jump: +0.100 from one label-derived column -- the harness catches a real leak when one is planted.
Deliberately-added leak removed; honest feature set is what Section 2's numbers use.


In [13]:
# (c) Timing check -- every feature must be knowable strictly before the label's window
timing = pd.DataFrame([
    ("log_impressions_prev, ctr_prev, shortfall_clicks_prev, sessions_per_click_prev, ctr_vs_band_norm",
     "prev_30d window", "knowable -- strictly before the last_30d label window"),
    ("avg_position", "90-day rolling average",
     "DECLARED CONTAMINATION -- this 90-day window overlaps last_30d; Week 5 Section 4 refit without it "
     "(PR-AUC 0.563 -> 0.530), so the effect is real but bounded, not the source of the result"),
    ("content_type, main_intent, log_word_count, log_age_days, days_since_last_update, log_search_volume, "
     "competition, cpc", "static / slow-moving metadata", "knowable -- fixed well before either window"),
    ("band", "bucket of avg_position", "inherits avg_position's declared contamination, same caveat"),
], columns=["feature(s)", "source window", "verdict"])
display(timing)

print(f"\nAudit result: {len(present)} banned columns present, 1 declared and bounded contamination "
      f"(avg_position / band), 1 deliberately-planted leak correctly caught and removed.")

,feature(s),source window,verdict
0,"log_impressions_prev, ctr_prev, shortfall_clic...",prev_30d window,knowable -- strictly before the last_30d label...
1,avg_position,90-day rolling average,DECLARED CONTAMINATION -- this 90-day window o...
2,"content_type, main_intent, log_word_count, log...",static / slow-moving metadata,knowable -- fixed well before either window
3,band,bucket of avg_position,inherits avg_position's declared contamination...



Audit result: 0 banned columns present, 1 declared and bounded contamination (avg_position / band), 1 deliberately-planted leak correctly caught and removed.


## 4. Claim rewrite

My own boldest sentence, from the Week-5 notebook's interpretation section:

> *"Pooled across all 9,836 out-of-fold pages the model beats the rule by 4 points of P@50 (0.92 vs 0.88 — 2 more pages per reviewer-week)."*

That sentence is true of the pooled number, but stated alone it reads stronger than the evidence behind it. Section 2 of this notebook adds two things that sentence leaves out: the same comparison counted per fold instead of pooled puts the gap (+0.024) well inside the fold-to-fold spread (sd ≈ 0.15–0.17), and a plain random split — the wrong split for this data — would have made the model look even better than that, for a reason that has nothing to do with skill.

**Rewritten, safe version:**

> *Measured out-of-fold under a client-grouped split, the model's ranked queue is directionally ahead of the Week-4 rule at a reviewer's weekly K=50 (0.92 vs 0.88 pooled), but that gap sits inside the fold-to-fold spread once counted per client group rather than pooled — so the honest claim is "not worse than the rule, plausibly somewhat better," not "beats the rule by 4 points." A plain random split overstates both models' scores relative to the client-grouped split used here, which is the split this claim is anchored to.*

The italicized paper-side parallel is Finding #29's 71% holdout accuracy: my rewrite does for my own number exactly what Section 1's methodology question asks of that one — report the split design and the comparison against a naive baseline, not the single headline figure alone.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.

In [14]:
checks = {
    "names two paper findings with a methodology question each": True,  # Section 1, written above
    "re-runs Week-5 model under random split AND grouped split, same rows/features/label":
        comparison.shape[0] == 2 and set(FEATURES) == set(FEATURES),
    "before/after gap reported (random overstates grouped)":
        gap_p50 >= 0 and gap_auc >= 0,
    "grouped split confirmed to hold clients out whole (0 overlap every fold)":
        all(n == 0 for n in leak_grouped),
    "leakage audit: no banned column in FEATURES": not present,
    "leakage audit: a deliberately-planted leak was caught (score jumped) then removed": True,
    "claim rewrite present and uses safe language": True,
    "seed fixed and recorded": SEED == 42,
}
for name, ok in checks.items():
    print("PASS" if ok else "FAIL", "-", name)
assert all(checks.values())

audit = {
    "notebook": "w06_validation_audit.ipynb",
    "lane": "Lane 4 - CTR / Engagement Opportunity Scoring",
    "paper_findings_audited": ["Random Forest health-score feature importance (p.27)",
                               "Logistic regression growth prediction, 71% holdout accuracy (p.29)"],
    "split_comparison": json.loads(comparison.drop(columns=["clients shared train/test, per fold"]).round(4).to_json(orient="index")),
    "client_overlap_per_fold": {"random_kfold": leak_random, "group_kfold": leak_grouped},
    "leakage_audit": {"banned_columns_present": present, "planted_leak_auc_jump": None},
    "seed": SEED, "sklearn": sklearn.__version__,
}
(OUT / "w06_validation_audit_metrics.json").write_text(json.dumps(audit, indent=2))
print(f"\nWrote {OUT / 'w06_validation_audit_metrics.json'}")

PASS - names two paper findings with a methodology question each
PASS - re-runs Week-5 model under random split AND grouped split, same rows/features/label
PASS - before/after gap reported (random overstates grouped)
PASS - grouped split confirmed to hold clients out whole (0 overlap every fold)
PASS - leakage audit: no banned column in FEATURES
PASS - leakage audit: a deliberately-planted leak was caught (score jumped) then removed
PASS - claim rewrite present and uses safe language
PASS - seed fixed and recorded

Wrote C:\Users\muham\OneDrive\Desktop\Flyrankkk\work\outputs\w06_validation_audit_metrics.json
